In [1]:
import sys
sys.path.append('/home/cloud/Desktop/abhi/VillainNet/')

In [2]:
from datasets import Dataset, PoisonDataset_TwoTuple
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def pil_loader(path: str):
        ''' Load a pill image'''
        # open path as file to avoid ResourceWarning (https://github.com/python-pillow/Pillow/issues/835)
        with open(path, "rb") as f:
            img = Image.open(f)
            return img.convert("RGB")

# TODO: specify the return type
def accimage_loader(path: str):
    import accimage  # type: ignore
    ''' acc images?'''
    try:
        return accimage.Image(path)
    except OSError:
        # Potentially a decoding problem, fall back to PIL.Image
        return pil_loader(path)
        
def default_loader(path: str):
    ''' Default image loader for an image from the path: path'''
    from torchvision import get_image_backend

    if get_image_backend() == "accimage":
        return accimage_loader(path)
    else:
        return pil_loader(path)

def build_train_transform(im_size=224):
    # image_size = [128, 160, 192, 224]
    image_size = im_size
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.33425963, 0.31757396, 0.3210416], std=[0.27581533, 0.27270334, 0.27360208]),
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def custom_collate(batch):
    """Ensures labels remain tuples when collating."""
    samples, labels = zip(*batch)  # Unzip batch
    samples = torch.stack(samples, dim=0)  # Stack images
    return samples, labels  # Keep labels as tuples


In [4]:
data_path = "../classification_datasets/GTSRB"
poison_data_path = "../classification_datasets_poisoned/GTSRB_RS/GTSRB_RS_10"

train_path = data_path + '/train/'
test_path = data_path + '/test/Images/'

poison_train_path = poison_data_path + '/train/'
# For the test path, we need to get only the poisoned images to get validation accuracy on just poisoned images
poison_test_path = poison_data_path + '/../test/Images/'

# train_dataset = PoisonDataset_TwoTuple(poison_train_path, poison_class=8,
#                                             poison_ext='.png', loader=default_loader, extensions=[".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"], transform=build_train_transform())
# train_loader_poison = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=28,
#                                                   pin_memory=True, collate_fn=custom_collate)
# print(train_loader_poison)
# print(len(train_loader_poison))
dataset_ = Dataset(data_path, train_path, test_path, poison_train_path, poison_test_path)
dataset_.calc_stats()

dataset_.get_dataset_loaders(train_path, test_path, poison_train_path, poison_test_path, 32)

len files: 39270

[0.33502219 0.31852884 0.32157661] [0.27621168 0.27341229 0.27398405]


In [6]:
def view_dataloader(test_loader):
    features, labels = next(iter(test_loader))
    figure = plt.figure(figsize=(8, 8))
    cols, rows = 3,3
    for i in range(1, cols * rows + 1):
        img = features[i-1]
        label = labels[i-1]
        figure.add_subplot(rows, cols, i)
        plt.title(f"Clean: {str(label[0])}, Poison: {str(label[1])}")
        plt.axis("off")
        plt.imshow(img.squeeze().permute(1, 2, 0))
    plt.show()

In [7]:
print("Poison Train Loader")
view_dataloader(dataset_.train_loader_poison)

Poison Train Loader


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/torch/utils/data/_utils/worker.py", line 302, in _worker_loop
    data = fetcher.fetch(index)
  File "/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    return self.collate_fn(data)
TypeError: poison_two_tuple_collate() takes 1 positional argument but 2 were given


In [ ]:
view_dataloader(dataset_.train_loader_clean)

In [ ]:
print("Clean Test Loader")
view_dataloader(dataset_.test_loader_clean)
print(dataset_.test_loader_clean.dataset)

In [ ]:
print("Poison Train Loader")
view_dataloader(dataset_.train_loader_poison)

In [ ]:
print("Poison Test Loader")
view_dataloader(dataset_.test_loader_poison)